In [ ]:
#!/usr/bin/env python
# coding: utf-8

# Heart Disease Prediction using Supervised Machine Learning
**Student Minor Project 1**
Dataset: Cleveland Heart Disease Dataset (UCI Repository)
Source: https://archive.ics.uci.edu/dataset/45/heart+disease

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, classification_report)

print("Libraries imported successfully!")

## 2. Load Dataset

In [ ]:
df = pd.read_csv('../data/heart.csv')
print(f"Dataset Shape: {df.shape}")
print("\nFirst 5 rows:")
df.head()

## 3. Dataset Overview

In [ ]:
print("Dataset Info:")
print(df.info())
print("\nStatistical Summary:")
df.describe()

## 4. Data Preprocessing

In [ ]:
# Check missing values
print("Missing Values:\n", df.isnull().sum())

# Check duplicates
print(f"\nDuplicate rows: {df.duplicated().sum()}")
df.drop_duplicates(inplace=True)
df.reset_index(drop=True, inplace=True)
print(f"Shape after removing duplicates: {df.shape}")

# Target distribution
print("\nTarget Distribution:")
print(df['target'].value_counts())
print(f"0 = No Heart Disease, 1 = Heart Disease")

## 5. Exploratory Data Analysis (EDA)

In [ ]:
sns.set_style('whitegrid')
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 5.1 Target distribution
df['target'].value_counts().plot(kind='bar', color=['#2196F3', '#F44336'], ax=axes[0,0])
axes[0,0].set_xticklabels(['No Disease', 'Heart Disease'], rotation=0)
axes[0,0].set_title('Target Class Distribution')
axes[0,0].set_ylabel('Count')

# 5.2 Age distribution
for t, label, color in [(0, 'No Disease', '#2196F3'), (1, 'Heart Disease', '#F44336')]:
    df[df['target']==t]['age'].plot(kind='kde', label=label, color=color, ax=axes[0,1])
axes[0,1].set_title('Age Distribution by Target')
axes[0,1].set_xlabel('Age')
axes[0,1].legend()

# 5.3 Chest pain type
pd.crosstab(df['cp'], df['target']).plot(kind='bar', color=['#2196F3', '#F44336'], ax=axes[1,0])
axes[1,0].set_xticklabels(['Typical', 'Atypical', 'Non-anginal', 'Asymptomatic'], rotation=20)
axes[1,0].set_title('Chest Pain Type vs Heart Disease')
axes[1,0].legend(['No Disease', 'Heart Disease'])

# 5.4 Max heart rate vs age scatter
for t, label, color in [(0, 'No Disease', '#2196F3'), (1, 'Heart Disease', '#F44336')]:
    subset = df[df['target']==t]
    axes[1,1].scatter(subset['age'], subset['thalach'], label=label, color=color, alpha=0.6, s=30)
axes[1,1].set_title('Age vs Max Heart Rate')
axes[1,1].set_xlabel('Age')
axes[1,1].set_ylabel('Max Heart Rate (thalach)')
axes[1,1].legend()

plt.suptitle('Exploratory Data Analysis — Heart Disease Dataset', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../results/eda_combined.png', dpi=150)
plt.show()

# 5.5 Correlation Heatmap
plt.figure(figsize=(11, 9))
sns.heatmap(df.corr(), annot=True, fmt='.2f', cmap='coolwarm', linewidths=0.5)
plt.title('Feature Correlation Heatmap', fontsize=14)
plt.tight_layout()
plt.savefig('../results/03_correlation_heatmap.png', dpi=150)
plt.show()

## 6. Feature Engineering & Train-Test Split

In [ ]:
X = df.drop('target', axis=1)
y = df['target']

print("Features:", list(X.columns))
print("Target: target")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"\nTrain size: {X_train.shape[0]} | Test size: {X_test.shape[0]}")

## 7. Feature Scaling

In [ ]:
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)
print("Feature scaling applied (StandardScaler)")

## 8. Model Training & Evaluation

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree':       DecisionTreeClassifier(random_state=42),
    'Random Forest':       RandomForestClassifier(n_estimators=100, random_state=42)
}

results = {}
for name, model in models.items():
    model.fit(X_train_sc, y_train)
    y_pred = model.predict(X_test_sc)
    results[name] = {
        'Accuracy':  round(accuracy_score(y_test, y_pred) * 100, 2),
        'Precision': round(precision_score(y_test, y_pred) * 100, 2),
        'Recall':    round(recall_score(y_test, y_pred) * 100, 2),
        'F1-Score':  round(f1_score(y_test, y_pred) * 100, 2),
    }
    print(f"\n{'='*40}")
    print(f"Model: {name}")
    print(classification_report(y_test, y_pred, target_names=['No Disease', 'Heart Disease']))

results_df = pd.DataFrame(results).T
print("\nSummary Table:")
print(results_df)

## 9. Best Model — Random Forest

In [ ]:
best_model = models['Random Forest']
y_pred_rf  = best_model.predict(X_test_sc)

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred_rf)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['No Disease', 'Heart Disease'],
            yticklabels=['No Disease', 'Heart Disease'])
plt.title('Random Forest — Confusion Matrix')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.savefig('../results/05_confusion_matrix.png', dpi=150)
plt.show()

# Model Comparison Chart
fig, ax = plt.subplots(figsize=(10, 5))
results_df.plot(kind='bar', ax=ax, color=['#4CAF50', '#2196F3', '#FF9800', '#9C27B0'])
ax.set_title('Model Comparison — Evaluation Metrics (%)')
ax.set_ylabel('Score (%)')
ax.set_xticklabels(results_df.index, rotation=15)
ax.legend(loc='lower right')
ax.set_ylim(50, 105)
plt.tight_layout()
plt.savefig('../results/06_model_comparison.png', dpi=150)
plt.show()

# Feature Importance
fi = pd.Series(best_model.feature_importances_, index=X.columns).sort_values(ascending=True)
fig, ax = plt.subplots(figsize=(8, 6))
fi.plot(kind='barh', color='#2196F3', ax=ax)
ax.set_title('Random Forest — Feature Importance')
plt.tight_layout()
plt.savefig('../results/07_feature_importance.png', dpi=150)
plt.show()

## 10. Save Best Model

In [ ]:
pickle.dump(best_model, open('../model/random_forest_model.pkl', 'wb'))
pickle.dump(scaler,     open('../model/scaler.pkl', 'wb'))
print("Model and scaler saved to /model/")

## 11. Conclusion

In [ ]:
print("""
CONCLUSION
==========
- Three supervised ML models were trained and evaluated on the Heart Disease dataset.
- Random Forest achieved the best performance with 84% accuracy, 83.72% precision,
  80% recall, and 81.82% F1-Score.
- Key predictive features: thal, ca, cp, thalach, and oldpeak.
- The model can assist clinicians in early heart disease risk screening.
""")